# Работа с табличными данными

## 1. Pandas

**Pandas** — это одна из самых популярных библиотек Python для работы с табличными данными. Она предоставляет удобные 
структуры данных:
* **Series** — одномерный массив с индексами, похожий на колонку в таблице;
* **DataFrame** — двумерная таблица, где каждая колонка может иметь свой тип данных.

![pandas](../production/AI_Data_Analytics.Project_3.ID_1577559/misc/images/pandas.jpeg)

С помощью Pandas можно: фильтровать строки и столбцы, группировать данные, агрегировать показатели, объединять таблицы, 
работать с пропущенными значениями и строить простые визуализации. Владение Pandas — базовый и необходимый навык для 
проведения EDA.

In [1]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Задание 1.1. Загрузка данных
1. Пройди по ссылке на хранилище с датасетом [2025 Kaggle Machine Learning & Data Science Survey](https://disk.360.yandex.ru/d/ATLDXjTVorMf3Q).
3. Скачай CSV-файлы с данными и сохрани их в папку `datasets`.
4. С помощью библиотеки `pandas` загрузи:
   * таблицу с выбором ответов (multiple choice) в переменную `multiple`,
   * таблицу со свободными ответами (free form) `freeform`.
5. Выведи размеры каждой таблицы.
   
> **Важно:** загружать датасеты на Git не нужно.

In [2]:
freeform = pd.read_csv("../assets/project3/2025/freeFormResponses.csv", low_memory=False)

In [3]:
multiple = pd.read_csv("../assets/project3/2025/multipleChoiceResponses.csv", low_memory=False)

In [4]:
freeform.shape

(23860, 35)

In [5]:
multiple.shape

(23860, 395)

### Задание 1.2. Словарь вопросов
В таблицах `freeform` и `multiple`  первая строка содержит текстовые формулировки вопросов (например: *'What is your gender? — Selected Choice'*, *'What is your age (# years)?'* и т. д.), а сами данные начинаются со второй строки.

1. Создай структуру `name2question`, которая для каждой колонки содержит словарь с ключами `question` и `type`. В ключе `question` хранится текст вопроса, а в ключе `type` — тип таблицы, из которой пришел вопрос: `'multiple'`, `'freeform'` или `'both'` (если колонка есть в обеих таблицах).
```
name2question['Q1']
{'question': 'What is your gender? - Selected Choice', 'source': 'multiple'}
```

2. Выведи значение для вопроса `Q1_OTHER_TEXT` из структуры `name2question`.
3. Удали первые строки из таблиц `freeform` и `multiple`.
4. Выведи размерности таблиц `freeform` и `multiple`.

In [6]:
# 1. Создаем структуру name2question
name2question = {}

# Получаем первую строку из каждой таблицы (тексты вопросов)
multiple_questions = multiple.iloc[0]
freeform_questions = freeform.iloc[0]

# Получаем множества колонок для определения общих
multiple_cols = set(multiple.columns)
freeform_cols = set(freeform.columns)
all_cols = multiple_cols.union(freeform_cols)

# Заполняем словарь для каждой колонки
for col in all_cols:
    question_text = None
    source = None
    
    # Определяем источник и текст вопроса
    in_multiple = col in multiple_cols
    in_freeform = col in freeform_cols
    
    if in_multiple and in_freeform:
        source = 'both'
        # Если колонка есть в обеих таблицах, берем вопрос из multiple (или можно из freeform)
        question_text = multiple_questions[col]
    elif in_multiple:
        source = 'multiple'
        question_text = multiple_questions[col]
    elif in_freeform:
        source = 'freeform'
        question_text = freeform_questions[col]
    
    name2question[col] = {
        'question': question_text,
        'source': source
    }

# 2. Выводим значение для Q1_OTHER_TEXT
print("Значение для Q1_OTHER_TEXT:")
print(name2question['Q1_OTHER_TEXT'])

# 3. Удаляем первые строки из таблиц
freeform = freeform.iloc[1:].reset_index(drop=True)
multiple = multiple.iloc[1:].reset_index(drop=True)

# 4. Выводим размерности таблиц
print("\nРазмерность таблицы freeform:", freeform.shape)
print("Размерность таблицы multiple:", multiple.shape)

Значение для Q1_OTHER_TEXT:
{'question': 'What is your gender? - Prefer to self-describe - Text', 'source': 'both'}

Размерность таблицы freeform: (23859, 35)
Размерность таблицы multiple: (23859, 395)


### Задание 1.3. Анализ общих колонок

В таблице `multiple` некоторые колонки содержат числовые коды ответов. Если значение отлично от `-1`, это означает, что респондент выбрал свой вариант ответа, и в таблице `freeform` для той же строки (с тем же индексом) должно присутствовать соответствующее текстовое значение в свободной форме.

Проверить возможность корректного объединения таблиц `multiple` и `freeform`.

1. Определи общие колонки:
   - Найди все колонки, которые присутствуют одновременно в таблицах `multiple` и `freeform`.
   - Выведи их количество.

2. Проведи анализ общих колонок:
   - Выбери 2–3 общие колонки для детального изучения (например, `Q1_OTHER_TEXT`, `Q6_OTHER_TEXT`).
   - Для каждой выбранной колонки:
     - Выведи частоты значений в таблице `multiple` (количество значений, отличных от `-1`).
     - Выведи частоты значений в таблице `freeform` (количество непустых текстовых значений).
     - Найди индексы строк, где в таблице `multiple` значение отлично от `-1`.
     - Найди индексы строк, где в таблице `freeform` присутствуют текстовые значения.
     - Сравни эти два набора индексов.

3. Напиши свои выводы по результатам анализа:
   - Можно ли корректно объединить эти две таблицы по индексам строк?
   - Если нет, приведи конкретные примеры, демонстрирующие проблему (выведи соответствующие строки из обеих таблиц).

In [46]:
freeform_columns = set(freeform.columns)
multiple_columns = set(multiple.columns)

In [47]:
over_lapper_columns = list(freeform_columns.intersection(multiple_columns))
len(over_lapper_columns)

35

***Результаты анализа:***

Просто так объединить нельзя

### Задание 1.4. Объединение таблиц

### Задание 1.4. Анализ пропущенных значений

Работа с пропущенными значениями — важная часть EDA (Exploratory Data Analysis). Пропущенные значения (missing values) могут возникать по разным причинам: респонденты могли пропустить вопрос, данные могли быть потеряны при сборе или обработке, или вопрос мог быть не применим к конкретному респонденту.

Анализ пропущенных значений помогает понять качество данных и принять решение о том, как с ними работать: удалить колонки или строки с большим количеством пропусков, заполнить пропуски средними значениями или другими методами импутации, или оставить их как есть.

В опросах часто встречаются вопросы, на которые никто не ответил — такие колонки полностью состоят из пропущенных значений. Такие колонки не несут полезной информации и могут быть удалены из анализа.

1. Используя метод `.isna()`, найди все колонки в таблице `multiple`, где люди ни разу не отвечали (все значения пропущены).
2. Используя метод `.isna()`, найди все колонки в таблице `freeform`, где люди ни разу не отвечали (все значения пропущены).
3. Выведи названия найденных колонок для каждой таблицы.


In [20]:
freeform["Q11_OTHER_TEXT"].value_counts()

Q11_OTHER_TEXT
I am a student                                                          49
Student                                                                 46
I'm a student                                                           15
student                                                                 13
i am a student                                                           5
                                                                        ..
I am a student. Why do you keep asking my work? I don't have a work.     1
Build prototype to get statistical calculations                          1
Visualizations                                                           1
Procure data that feeds into machine learning work                       1
student and I dont work                                                  1
Name: count, Length: 399, dtype: int64

In [21]:
multiple["Q11_OTHER_TEXT"].value_counts()

Q11_OTHER_TEXT
-1     23315
16        49
27        46
23        15
5         13
       ...  
135        1
134        1
133        1
132        1
397        1
Name: count, Length: 400, dtype: int64

In [22]:
# responses[responses["Q11_OTHER_TEXT"].isin(["16"])][["Q11_OTHER_TEXT"]]

In [23]:
freeform[freeform["Q11_OTHER_TEXT"] == "I am a student"]["Q11_OTHER_TEXT"].head(5)

276     I am a student
508     I am a student
1716    I am a student
2811    I am a student
3415    I am a student
Name: Q11_OTHER_TEXT, dtype: object

In [24]:
multiple[multiple["Q11_OTHER_TEXT"] == "16"]["Q11_OTHER_TEXT"].head(5)

476    16
520    16
621    16
674    16
769    16
Name: Q11_OTHER_TEXT, dtype: object

In [25]:
responses[responses["Q11_OTHER_TEXT"].isin(["I am a student", "16"])][["Q11_OTHER_TEXT"]]

,Q11_OTHER_TEXT
276,I am a student
476,16
508,I am a student
520,16
621,16
...,...
22945,I am a student
23137,I am a student
23424,16
23591,I am a student


In [27]:
responses["Q11_OTHER_TEXT"].value_counts()

Q11_OTHER_TEXT
-1                                                                                                     22778
I am a student                                                                                            49
16                                                                                                        49
Student                                                                                                   46
27                                                                                                        46
                                                                                                       ...  
156                                                                                                        1
I am a student, we have courses related to ml, hard to answer questions job related as a student ;)        1
I am a Student                                                                                             1
Rese

In [49]:
multiple["Q6_OTHER_TEXT"].loc[[773, 832]]

773    -1
832    -1
Name: Q6_OTHER_TEXT, dtype: object

In [50]:
freeform[freeform["Q6_OTHER_TEXT"] == "Professor"]["Q6_OTHER_TEXT"].head(5)

773     Professor
832     Professor
866     Professor
909     Professor
1013    Professor
Name: Q6_OTHER_TEXT, dtype: object

In [51]:
responses["Q6_OTHER_TEXT"].loc[[773, 832]]

773    Professor
832    Professor
Name: Q6_OTHER_TEXT, dtype: object

In [57]:
responses["Q6_OTHER_TEXT"].value_counts().sum() - 21278

2581

In [41]:
multiple["Q6_OTHER_TEXT"].value_counts()

Q6_OTHER_TEXT
-1     22537
7         49
41        42
43        39
28        17
       ...  
341        1
342        1
343        1
344        1
861        1
Name: count, Length: 864, dtype: int64

In [27]:
responses.columns[responses.isna().sum() == 23859]

Index(['Q13_Part_14', 'Q14_Part_10', 'Q16_Part_17', 'Q19_Part_18',
       'Q21_Part_12', 'Q27_Part_19', 'Q28_Part_42', 'Q29_Part_27',
       'Q30_Part_24', 'Q36_Part_12', 'Q38_Part_19', 'Q38_Part_20'],
      dtype='object')

In [30]:
name2question["Q38_Part_19"]

{'question': 'Who/what are your favorite media sources that report on data science topics? (Select all that apply) - Selected Choice - Towards Data Science Blog',
 'source': 'multiple'}

In [31]:
name2question["Q38_Part_20"]

{'question': 'Who/what are your favorite media sources that report on data science topics? (Select all that apply) - Selected Choice - Analytics Vidhya Blog',
 'source': 'multiple'}

In [25]:
responses

,Time from Start to Finish (seconds),Q1,Q1_OTHER_TEXT,Q2,Q3,Q4,Q5,Q6,Q6_OTHER_TEXT,Q7,...,Q49_OTHER_TEXT,Q50_Part_1,Q50_Part_2,Q50_Part_3,Q50_Part_4,Q50_Part_5,Q50_Part_6,Q50_Part_7,Q50_Part_8,Q50_OTHER_TEXT
1,710,Female,-1,45-49,United States of America,Doctoral degree,Other,Consultant,-1,Other,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
2,434,Male,-1,30-34,Indonesia,Bachelor’s degree,Engineering (non-computer focused),Other,0,Manufacturing/Fabrication,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
3,718,Female,-1,30-34,United States of America,Master’s degree,"Computer science (software engineering, etc.)",Data Scientist,-1,I am a student,...,-1,NaN,Too time-consuming,NaN,NaN,NaN,NaN,NaN,NaN,-1
4,621,Male,-1,35-39,United States of America,Master’s degree,"Social sciences (anthropology, psychology, soc...",Not employed,-1,NaN,...,-1,NaN,NaN,Requires too much technical knowledge,NaN,Not enough incentives to share my work,NaN,NaN,NaN,-1
5,731,Male,-1,22-24,India,Master’s degree,Mathematics or statistics,Data Analyst,-1,I am a student,...,-1,NaN,Too time-consuming,NaN,NaN,Not enough incentives to share my work,NaN,NaN,NaN,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23855,575,Male,-1,45-49,France,Doctoral degree,"Computer science (software engineering, etc.)",Chief Officer,-1,Computers/Technology,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
23856,131,Female,-1,25-29,Turkey,Master’s degree,Engineering (non-computer focused),NaN,-1,NaN,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
23857,370,Male,-1,22-24,Turkey,Master’s degree,"Computer science (software engineering, etc.)",Software Engineer,-1,Computers/Technology,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
23858,36,Male,-1,25-29,United Kingdom of Great Britain and Northern I...,NaN,NaN,NaN,-1,NaN,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1


In [21]:
# 1. Находим колонки в multiple, где все значения пропущены (100% пропусков)
# Подсчитываем пропущенные значения в каждой колонке
missing_counts_multiple = responses.isna().sum()
# Находим колонки, где количество пропусков равно общему количеству строк
completely_empty_multiple = missing_counts_multiple[missing_counts_multiple == len(multiple)].index.tolist()

print("Колонки в таблице multiple, где люди ни разу не отвечали:")
print(completely_empty_multiple)
print(f"Количество таких колонок: {len(completely_empty_multiple)}")
print()

# 2. Находим колонки в freeform, где все значения пропущены (100% пропусков)
# Подсчитываем пропущенные значения в каждой колонке
missing_counts_freeform = freeform.isna().sum()
# Находим колонки, где количество пропусков равно общему количеству строк
completely_empty_freeform = missing_counts_freeform[missing_counts_freeform == len(freeform)].index.tolist()

print("Колонки в таблице freeform, где люди ни разу не отвечали:")
print(completely_empty_freeform)
print(f"Количество таких колонок: {len(completely_empty_freeform)}")


Колонки в таблице multiple, где люди ни разу не отвечали:
[]
Количество таких колонок: 0

Колонки в таблице freeform, где люди ни разу не отвечали:
[]
Количество таких колонок: 0


In [18]:
missing_counts = responses.isna().sum()
missing_counts

Time from Start to Finish (seconds)        0
Q1                                         0
Q1_OTHER_TEXT                              0
Q2                                         0
Q3                                         0
                                       ...  
Q50_Part_5                             20290
Q50_Part_6                             22800
Q50_Part_7                             21359
Q50_Part_8                             23339
Q50_OTHER_TEXT                             0
Length: 395, dtype: int64

In [24]:
top_20_missing = missing_counts.sort_values(ascending=False).head(20)
print("Топ-20 колонок с наибольшим количеством пропусков:")
print(top_20_missing)

Топ-20 колонок с наибольшим количеством пропусков:
Q21_Part_12    23859
Q29_Part_27    23859
Q14_Part_10    23859
Q27_Part_19    23859
Q38_Part_20    23859
Q38_Part_19    23859
Q19_Part_18    23859
Q28_Part_42    23859
Q16_Part_17    23859
Q36_Part_12    23859
Q13_Part_14    23859
Q30_Part_24    23859
Q28_Part_22    23842
Q28_Part_24    23828
Q29_Part_16    23821
Q29_Part_25    23806
Q29_Part_14    23800
Q29_Part_24    23798
Q30_Part_15    23798
Q29_Part_23    23793
dtype: int64


In [25]:
missing_percentage = (missing_counts / len(responses)) * 100
print("Доля пропущенных значений (в процентах) для каждой колонки:")
print(missing_percentage)
print()

Доля пропущенных значений (в процентах) для каждой колонки:
Time from Start to Finish (seconds)     0.000000
Q1                                      0.000000
Q1_OTHER_TEXT                           0.000000
Q2                                      0.000000
Q3                                      0.000000
                                         ...    
Q50_Part_5                             85.041284
Q50_Part_6                             95.561423
Q50_Part_7                             89.521774
Q50_Part_8                             97.820529
Q50_OTHER_TEXT                          0.000000
Length: 395, dtype: float64



In [26]:
columns_over_50_percent = (missing_percentage > 50).sum()
print(f"Количество колонок с более чем 50% пропущенных значений: {columns_over_50_percent}")
print()

Количество колонок с более чем 50% пропущенных значений: 314



In [27]:
rows_with_missing = responses.isna().any(axis=1).sum()
print(f"Количество строк с хотя бы одним пропуском: {rows_with_missing}")

Количество строк с хотя бы одним пропуском: 23859


In [34]:
responses.columns[responses.isna().sum() == len(responses)]

Index(['Q13_Part_14', 'Q14_Part_10', 'Q16_Part_17', 'Q19_Part_18',
       'Q21_Part_12', 'Q27_Part_19', 'Q28_Part_42', 'Q29_Part_27',
       'Q30_Part_24', 'Q36_Part_12', 'Q38_Part_19', 'Q38_Part_20'],
      dtype='object')

In [36]:
responses["Q13_Part_14"].isna().sum()

23859

### Задание 1.4. Очистка респондентов

В данных есть информация о времени заполнения опроса. Логично предположить, что если респондент отвечал **слишком быстро** или, наоборот, **слишком долго**, такие ответы могут быть недостоверными. Для очистки данных воспользуемся методом выявления выбросов на основе **box plot**.

**Box plot (ящик с усами)** — это способ визуализации распределения данных. Внутри прямоугольника (ящика) находятся значения от 1-го до 3-го квартиля (Q1 и Q3), линия внутри — это медиана. «Усы» обычно определяются через **интерквартильный размах (IQR)**, который равен:

$$IQR = Q3 - Q1$$
Значения ниже $Q1 - 1.5 \times IQR$ и выше $Q3 + 1.5 \times IQR$ считаются выбросами.

1. Построй box plot для времени заполнения опроса. Ограничь ось **X** от 0 до 5000 секунд.
2. Рассчитай интерквартильный размах (IQR).
3. Выведи нижнюю и верхнюю границы интервала для выявления выбросов.
4. Так как нижняя граница получилась отрицательной, будем априорно считать минимальное адекватное время заполнения равным **2 минутам (120 секунд)**.
5. Отфильтруй DataFrame, оставив только респондентов с временем заполнения от 120 секунд до верхней границы.
6. Выведи размерность очищенного датасета.

In [38]:
responses["Time from Start to Finish (seconds)"] = responses["Time from Start to Finish (seconds)"].astype(int)

In [39]:
multiple["Time from Start to Finish (seconds)"].plot.box()
plt.ylim(0, 5000)
plt.show();

TypeError: no numeric data to plot

In [20]:
Q1 = np.percentile(responses["Time from Start to Finish (seconds)"], 25)
Q3 = np.percentile(responses["Time from Start to Finish (seconds)"], 75)
IQR = Q3 - Q1
print(IQR)

1272.0


In [21]:
upper = Q3+1.5*IQR
print("Upper Bound:", upper)

lower = Q1-1.5*IQR
print("Lower Bound:", lower)

Upper Bound: 3750.0
Lower Bound: -1338.0


In [22]:
responses = responses[
    (responses["Time from Start to Finish (seconds)"] < upper) & 
    (responses["Time from Start to Finish (seconds)"] > 120)
]

In [23]:
responses.shape

(18696, 395)

### Задание 1.5. Не популярный гендер
Обработай результаты опроса по вопросу `What is your gender?`:
1. Объедини ответы из обеих колонок
2. Удали у всех значений ведущие и завершающие пробелы
3. Приведи все ответы к нижнему регистру
4. Удали ответы, содержащие числа
5. Выведи **топ-6 самых популярных** вариантов ответа

In [41]:
multiple["Q1"].value_counts()

Q1
Male                                      19430
Female                                     4010
Prefer not to say                           340
Prefer to self-describe                      79
What is your gender? - Selected Choice        1
Name: count, dtype: int64

In [24]:
name2question["Q1"]

'What is your gender? - Selected Choice'

In [25]:
name2question["Q1_OTHER_TEXT"]

'What is your gender? - Prefer to self-describe - Text'

In [26]:
q1_res = responses["Q1"].str.lower().str.strip().value_counts()
mask = q1_res.index.astype(str).str.contains(r"\d")
q1_res = q1_res[~mask]

q1_res_text = responses["Q1_OTHER_TEXT"].str.lower().str.strip().value_counts()
mask = q1_res_text.index.astype(str).str.contains(r"\d")
q1_res_text = q1_res_text[~mask]

In [27]:
a = pd.concat([q1_res, q1_res_text])

In [28]:
a.sort_values(ascending=False).index

Index(['male', 'female', 'prefer not to say', 'prefer to self-describe',
       'attack helicopter', 'non-binary', 'agender', 'female', 'nonbinary',
       'male', 'transgender', 'transgender female', 'human', 'kaggle',
       'a little sunshine. :)', 'non binary', 'm', 'unclear', 'ai',
       'male and female are sexes not gender. gender is a regressive set of stereotypes associated with our sex. ask what sex we are for demographic purposes, if that is what is important.',
       'my sex is male (no idea about "gender")', 'bot', 'mard',
       'hetero flexible', 'no binary', 'nunya', 'shemale', 'barn owl',
       'gelatinous blob', 'fhjdfjhdfhj', 'apache helicopter',
       'male but asking this at the begining of surveys is going to cause priming and distort your answers. google it. 😀',
       'gaseous', 'trollogender', 'megatron', 'bigender', 'testosterone',
       'neutrois', 'bisexual', 'helicopter', 'jedi', 'surly tomato',
       'baba yaga', 'transgender-fluid transracial black 

In [29]:
responses.to_csv("../assets/project4/kaggle_survey_2025_responses_v1.csv")

In [30]:
with open("../assets/project4/name2question.json", 'w') as fout:
    json.dump(name2question, fout, indent=4)